In [1]:
!pip install tensorflow

  Using cached tensorflow-2.20.0-cp311-cp311-win_amd64.whl.metadata (4.6 kB)
Using cached tensorflow-2.20.0-cp311-cp311-win_amd64.whl (331.8 MB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\user\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\ext\\filters\\fault_injection\\fault_injection_service_config_parser.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip show tensorflow

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Read the CSV file into a pandas DataFrame
df = pd.read_csv("seattle-weather.csv")

# Display the first 5 rows of the DataFrame
display(df.head())

# Print the concise summary of the DataFrame
df.info()

# Display descriptive statistics of the DataFrame
display(df.describe())

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1461 non-null   object 
 1   precipitation  1461 non-null   float64
 2   temp_max       1461 non-null   float64
 3   temp_min       1461 non-null   float64
 4   wind           1461 non-null   float64
 5   weather        1461 non-null   object 
dtypes: float64(4), object(2)
memory usage: 68.6+ KB


,precipitation,temp_max,temp_min,wind
count,1461.000000,1461.000000,1461.000000,1461.000000
mean,3.029432,16.439083,8.234771,3.241136
std,6.680194,7.349758,5.023004,1.437825
min,0.000000,-1.600000,-7.100000,0.400000
25%,0.000000,10.600000,4.400000,2.200000
50%,0.000000,15.600000,8.300000,3.000000
75%,2.800000,22.200000,12.200000,4.000000
max,55.900000,35.600000,18.300000,9.500000


## Data Preprocessing

In [3]:
df['date'] = pd.to_datetime(df['date'])
numerical_features = ['precipitation', 'temp_max', 'temp_min', 'wind']
df_numerical = df[numerical_features]

In [4]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_numerical)

def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:(i + sequence_length)])
        y.append(data[i + sequence_length])
    return np.array(X), np.array(y)

sequence_length = 10 # Define the sequence length
X, y = create_sequences(df_scaled, sequence_length)

In [7]:
X.shape, y.shape

((1451, 10, 4), (1451, 4))

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Creation

In [10]:
!pip install tensorflow

  Using cached tensorflow-2.20.0-cp311-cp311-win_amd64.whl.metadata (4.6 kB)
Using cached tensorflow-2.20.0-cp311-cp311-win_amd64.whl (331.8 MB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\user\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\ext\\filters\\fault_injection\\fault_injection_service_config_parser.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
!pip show tensorflow

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential()
model.add(LSTM(units=50, return_sequences=False, input_shape=(sequence_length, df_numerical.shape[1])))
model.add(Dense(units=df_numerical.shape[1]))

model.summary()

ModuleNotFoundError: No module named 'tensorflow.python'

## Model compilation


Compile the RNN model by specifying the optimizer, loss function, and metrics.


In [9]:
from tensorflow.keras.optimizers import Adam

model.compile(optimizer=Adam(), loss='mean_squared_error', metrics=['mean_absolute_error'])

## Model training


Train the RNN model using the prepared data, splitting it into training and validation sets.


In [10]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.1336 - mean_absolute_error: 0.2730 - val_loss: 0.0186 - val_mean_absolute_error: 0.0963
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0191 - mean_absolute_error: 0.0979 - val_loss: 0.0149 - val_mean_absolute_error: 0.0891
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0159 - mean_absolute_error: 0.0915 - val_loss: 0.0133 - val_mean_absolute_error: 0.0842
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0147 - mean_absolute_error: 0.0872 - val_loss: 0.0129 - val_mean_absolute_error: 0.0832
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0142 - mean_absolute_error: 0.0859 - val_loss: 0.0130 - val_mean_absolute_error: 0.0830
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0136 - mean_absolute_error: 0.0846 - val_loss: 0.0127 - val_mean_absolute_error: 0.0837
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0140 - mean_absolute_error: 0.0846 - val_loss: 0.01

## Model evaluation




In [11]:
evaluation_results = model.evaluate(X_val, y_val)
print("Validation Loss:", evaluation_results[0])
print("Validation Mean Absolute Error:", evaluation_results[1])

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0117 - mean_absolute_error: 0.0753
Validation Loss: 0.010812714695930481
Validation Mean Absolute Error: 0.07309027761220932




### Data Analysis Key Findings

*   The initial attempt to load "data/GOOG.csv" failed; the analysis proceeded using "seattle-weather.csv".
*   The "seattle-weather.csv" dataset contains 1461 entries and 6 columns, with no missing values.
*   Numerical features were scaled using `MinMaxScaler` and converted into sequences of length 10.
*   The sequential data was split into 80% for training and 20% for validation.
*   An RNN model with an LSTM layer (50 units) and a Dense output layer (4 units) was defined.
*   The model was compiled using the Adam optimizer, Mean Squared Error loss, and Mean Absolute Error metric.
*   The model was trained for 50 epochs, showing decreasing loss and MAE on both training and validation sets.
*   On the validation set, the model achieved a Validation Loss of 0.0108 and a Validation Mean Absolute Error of 0.0731.
*   Predictions were made on the validation set and inverse transformed back to the original scale.

